# Demand & Inventory EDA

Explores the synthetic demand dataset before modeling.
Run `python data/generate_data.py` first to produce `data/demand.csv` and `data/events.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR = Path('../data')

demand = pd.read_csv(DATA_DIR / 'demand.csv', parse_dates=['week_start'])
events = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['date'])

print(f'Demand rows: {len(demand):,}')
print(f'SKUs: {demand["sku"].nunique()}')
print(f'Date range: {demand["week_start"].min().date()} → {demand["week_start"].max().date()}')
print(f'Events: {len(events)}')
demand.head()

## Weekly demand by SKU

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
fig.suptitle('Weekly Demand by SKU', fontsize=14, fontweight='bold', y=1.01)

skus = demand['sku'].unique()
colors = ['#2563eb', '#16a34a', '#dc2626', '#d97706', '#7c3aed']

for ax, sku, color in zip(axes, skus, colors):
    sku_df = demand[demand['sku'] == sku].sort_values('week_start')
    ax.plot(sku_df['week_start'], sku_df['units_sold'], color=color, linewidth=1.5, label=sku)
    
    # Mark event weeks
    event_weeks = sku_df[sku_df['event_attendance'] > 0]
    ax.scatter(event_weeks['week_start'], event_weeks['units_sold'],
               color='#f59e0b', s=20, zorder=5, alpha=0.8)
    
    ax.set_ylabel(sku, fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

axes[-1].set_xlabel('Week')
fig.text(0.02, 0.5, 'Units Sold', va='center', rotation='vertical', fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/demand_by_sku.png', dpi=150, bbox_inches='tight')
plt.show()
print('Yellow dots = event weeks')

## Event lift analysis

How much does each event type lift demand above the weekly baseline?

In [ ]:
# Tag event vs. non-event weeks
demand['has_event'] = demand['event_attendance'] > 0
demand['att_bucket'] = pd.cut(
    demand['event_attendance'],
    bins=[-1, 0, 15_000, 40_000, 200_000],
    labels=['No event', 'Small (<15K)', 'Medium (15–40K)', 'Large (>40K)']
)

lift = (
    demand.groupby(['sku', 'att_bucket'])['units_sold']
    .mean()
    .unstack('att_bucket')
    .assign(**{
        'lift_small':  lambda d: (d['Small (<15K)']  / d['No event'] - 1) * 100,
        'lift_medium': lambda d: (d['Medium (15–40K)'] / d['No event'] - 1) * 100,
        'lift_large':  lambda d: (d['Large (>40K)']  / d['No event'] - 1) * 100,
    })
    [['lift_small', 'lift_medium', 'lift_large']]
    .rename(columns={'lift_small': 'Small', 'lift_medium': 'Medium', 'lift_large': 'Large'})
)

ax = lift.plot(kind='bar', figsize=(10, 5), colormap='RdYlGn', rot=30, width=0.7)
ax.set_title('Average Demand Lift by Event Size and SKU (%)', fontsize=13)
ax.set_ylabel('Lift vs. no-event baseline (%)')
ax.axhline(0, color='#333', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/event_lift_by_sku.png', dpi=150, bbox_inches='tight')
plt.show()
lift.round(1)

## Seasonality decomposition (Beverages)

In [ ]:
from statsmodels.tsa.seasonal import STL

bev = demand[demand['sku'] == 'Beverages'].sort_values('week_start').set_index('week_start')['units_sold']
stl = STL(bev, period=52, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [('Observed', bev), ('Trend', stl.trend), ('Seasonal', stl.seasonal), ('Residual', stl.resid)]
colors_stl = ['#1e40af', '#15803d', '#b45309', '#6b21a8']

for ax, (label, series), color in zip(axes, components, colors_stl):
    ax.plot(series.index, series.values, color=color, linewidth=1.4)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.suptitle('STL Decomposition — Beverages (weekly)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation matrix

In [ ]:
import matplotlib.cm as cm

pivot = demand.pivot_table(index='week_start', columns='sku', values='units_sold')
corr  = pivot.corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.index)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

ax.set_title('SKU Demand Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()